In [7]:
## IMPORTS
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = '/content/drive/MyDrive/TFM_NoeliaGarciaGarcia/Pipeline'
else:
    BASE_PATH = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))

import pandas as pd
import numpy as np
from scipy import signal

print(f"Entorno: {'Colab' if IN_COLAB else 'Local'}")
print(f"BASE_PATH: {BASE_PATH}")

Entorno: Local
BASE_PATH: g:\Mi unidad\TFM_NoeliaGarciaGarcia\Pipeline


# INFORMACIÓN

## UNIDADES

Las unidades utilizadas en este análisis son las siguientes:

*   **O2**: umol/L
*   **Velocidades (vx, vy, vz)**: cm/s
*   **Presión (pres)**: mbar (milibares)
*   **Temperatura (temp)**: °C (grados Celsius)
*   **Hora (hour)**: Horas desde el inicio del registro (no es un timestamp tradicional, sino tiempo relativo).

## CÁLCULO

El flujo turbulento de O₂ se calcula mediante la siguiente fórmula:

```
FLUX = mean(w' * O2')
```

Donde:
*   `w'` = fluctuación de velocidad vertical después de detrending (unidades: `cm/s`)
*   `O2'` = fluctuación de oxígeno después de detrending. Note que el O₂ original en `umol/L` se convierte a `umol/m³` antes de aplicar el detrending. (unidades: `umol/m³`)

#### Unidades

1.  **Flujo Bruto (FLUX)**:
    Las unidades resultantes de la multiplicación `w' * O2'` son:
    ```
    (cm/s) * (umol/m³) = umol m⁻² s⁻¹
    ```

2.  **Conversión a milimoles (mmol)**:
    Para expresar el flujo en `mmol m⁻² s⁻¹`, dividimos el flujo bruto (en `umol m⁻² s⁻¹`) por `1000`:
    ```
    mmol m⁻² s⁻¹ = (umol m⁻² s⁻¹) / 1000
    ```

3.  **Flujo integrado por día**:
    Para tener el flujo integrado por día, se realiza la siguiente conversión:
    ```
    mmol m⁻² day⁻¹ = mmol m⁻² s⁻¹ * 86400
    ```

# CARGA DE DATOS

In [8]:
# Cargar dataset completo
df = pd.read_csv(os.path.join(BASE_PATH, "DATA", "RAW", "df_master.csv"), sep=r",")

In [9]:
# Cargar dataset limpio
df_clean = pd.read_csv(os.path.join(BASE_PATH, "DATA", "CLEAN", "df_clean.csv"), sep=r",")

# SHIFT

## EXPLICACIÓN

### ¿Por qué hace falta un shift?

En teoría, para calcular el flujo podríamos multiplicar directamente `w'[t] * O2'[t]`. Sin embargo, en datos reales puede existir un pequeño desfase temporal entre la señal de velocidad vertical y la señal de oxígeno. Este desfase se debe a una separación física entre sensores.

Por eso no calculamos el flujo solo con `shift = 0`, sino que probamos varios desplazamientos entre ambas señales.

### Importante: el shift no está en Hz

La frecuencia de muestreo sí está en Hz.

Por ejemplo:

`fs = 8 Hz` significa `8 muestras por segundo`.

Pero el shift no se expresa en Hz. El shift es un desplazamiento entre señales y se expresa en:

*   Muestras
*   O, de forma equivalente, segundos

En este pipeline usamos el shift en **MUESTRAS**.

Por tanto:

*   `shift = -8` significa `desplazar -8 muestras`.
*   `shift = 32` significa `desplazar 32 muestras`.

Si `fs = 8 Hz`, entonces cada muestra dura `1 / fs = 1 / 8 = 0.125 segundos`.

Por eso:

*   `shift = -8 muestras` -> `-8 / 8 = -1 segundo`
*   `shift = 0 muestras` -> `0 segundos`
*   `shift = 32 muestras` -> `32 / 8 = 4 segundos`

En resumen, al probar shifts desde -8 hasta 32 muestras, estamos probando desfases temporales desde -1 hasta 4 segundos cuando `fs = 8 Hz`.

### Rango de shifts que se prueba

En este caso probamos:

```
possible_shifts = range(-8, 33)
```

Esto incluye todos los shifts enteros desde -8 hasta 32: `-8, -7, -6, ..., -1, 0, 1, ..., 30, 31, 32`.

Usamos `33` como límite superior porque en Python `range(a, b)` incluye `a`, pero no incluye `b`.

### Convención del signo del shift

En esta implementación usamos la siguiente convención:

*   **`shift > 0`:**
    Se compara `w[shift:]` con `O2[:-shift]`.

    Por ejemplo, para `shift = 2`:

    `w[2] * O2[0]`
    `w[3] * O2[1]`
    `w[4] * O2[2]`
    ...

    Esto representa un caso en el que la señal de `O2` está retrasada respecto a la velocidad vertical `w`.

*   **`shift < 0`:**
    Se compara `w[:-abs(shift)]` con `O2[abs(shift):]`.

    Por ejemplo, para `shift = -2`:

    `w[0] * O2[2]`
    `w[1] * O2[3]`
    `w[2] * O2[4]`
    ...

    Esto representa el caso contrario: `O2` está adelantada respecto a `w`.

*   **`shift = 0`:**
    Se comparan las dos señales sin desplazar:

    `w[0] * O2[0]`
    `w[1] * O2[1]`
    `w[2] * O2[2]`
    ...

### Selección del shift óptimo

Para cada ventana temporal:

1.  Se aplica detrending lineal a `vz` y `O2`.
2.  Se prueban todos los shifts entre -8 y 32 muestras.
3.  Para cada shift se calcula un flujo candidato:

    `FLUX_candidate = mean(w'_shifted * O2'_shifted)`

4.  Se selecciona el shift que produce el flujo óptimo.

En este caso, usamos como criterio el flujo más grande en valor absoluto (sea positivo o negativo): `criterio = "max_abs"`.


Por tanto, para cada ventana nos quedamos con:

*   El flujo más grande encontrado dentro del rango de shifts.
*   El shift que produjo ese flujo.

## FUNCIÓN

In [10]:
def covarianza_con_shift(w_block, o2_block, shift_samples):
    """
    Calcula el flujo candidato entre w' y O2' para un shift dado.

    w_block:
        Velocidad vertical detrended, w', en m/s.

    o2_block:
        Oxígeno detrended, O2', en mmol/m³.

    shift_samples:
        Shift expresado en muestras, no en Hz ni en segundos.

    Unidades del resultado:
        m/s * mmol/m³ = mmol m⁻² s⁻¹
    """

    w_block = np.asarray(w_block, dtype=float)
    o2_block = np.asarray(o2_block, dtype=float)

    if abs(shift_samples) >= len(w_block):
        return np.nan

    if shift_samples > 0:
        w_s = w_block[shift_samples:]
        o2_s = o2_block[:-shift_samples]

    elif shift_samples < 0:
        s = abs(shift_samples)
        w_s = w_block[:-s]
        o2_s = o2_block[s:]

    else:
        w_s = w_block
        o2_s = o2_block

    valid = np.isfinite(w_s) & np.isfinite(o2_s)

    if valid.sum() < 30:
        return np.nan

    return np.mean(w_s[valid] * o2_s[valid])

# IMPLEMENTACIÓN

## PARÁMETROS

In [11]:
# Lectura de parámetros desde config.xlsx
config = pd.read_excel(os.path.join(BASE_PATH, "NOTEBOOKS", "config.xlsx"), sheet_name="config", index_col="parameter")["value"]
fs = int(config["fs"])
window_minutes = int(config["window_minutes"])
n_samples = fs * 60 * window_minutes

SECONDS_PER_DAY = 86400
HOURS_PER_DAY = 24

smooth_window = 30

apply_shift = bool(config["apply_shift"])
shift_min_samples = int(config["shift_min_samples"])
shift_max_samples = int(config["shift_max_samples"])

print(f"fs={fs}, window_minutes={window_minutes}, n_samples={n_samples}")
print(f"apply_shift={apply_shift}, shift_min_samples={shift_min_samples}, shift_max_samples={shift_max_samples}")

fs=8, window_minutes=15, n_samples=7200
apply_shift=True, shift_min_samples=-8, shift_max_samples=32


## CÁLCULO

In [12]:
def procesar_dataset_completo(
    df
):
    """
    Calcula el flujo de O2 por ventanas temporales usando un dataset limpio.


    Columnas esperadas en df:
        hour : horas desde el inicio del registro
        vz   : velocidad vertical, m/s
        O2   : oxígeno, mmol/m³
        temp : temperatura, ºC
        pres : presión, dbar

    Columnas opcionales procedentes de la limpieza:
        window_id      : identificador de ventana

    Parámetros
    ----------
    df : pandas.DataFrame
        Dataset completo.

    fs : int o float
        Frecuencia de muestreo en Hz.

    window_minutes : int o float
        Duración de cada ventana en minutos.

    smooth_window : int
        Número de ventanas usadas para suavizar el flujo.

    shift_min_samples : int
        Shift mínimo en muestras.

    shift_max_samples : int
        Shift máximo en muestras.

    Devuelve
    --------
    df_final : pandas.DataFrame
        Tabla con una fila por ventana.
    """

    # ============================================================
    # 0. Preparar dataset
    # ============================================================
    # Copiamos el dataframe para no modificar el original.
    # Ordenamos por tiempo para asegurar que las ventanas son temporales.

    dfp = df.copy().sort_values("hour").reset_index(drop=True)

    # ============================================================
    # 1. Tamaño de cada ventana
    # ============================================================

    total_samples = len(dfp)
    n_blocks = total_samples // n_samples

    if n_blocks == 0:
        return pd.DataFrame()

    limit = n_blocks * n_samples

    # Nos quedamos solo con las muestras que forman ventanas completas.
    # Las muestras sobrantes del final se descartan porque no forman
    # una ventana completa de window_minutes minutos.

    dfp = dfp.iloc[:limit].copy()

    # ============================================================
    # 2. Crear identificador de ventana
    # ============================================================
    # Si el dataframe ya trae window_id desde la limpieza, lo conservamos.
    # Si no existe, lo creamos.

    if "window_id" not in dfp.columns:
        dfp["window_id"] = np.repeat(
            np.arange(n_blocks),
            n_samples
        )

    # ============================================================
    # 3. Extraer variables
    # ============================================================
    # IMPORTANTE:
    # Aquí asumimos que las conversiones ya están hechas antes:
    #
    #     df["vz"] en m/s
    #     df["O2"] en mmol/m³
    #
    # Por tanto:
    #
    #     m/s * mmol/m³ = mmol m⁻² s⁻¹

    hour = dfp["hour"].to_numpy(dtype=np.float64)
    vx = dfp["vx"].to_numpy(dtype=np.float64)
    vy = dfp["vy"].to_numpy(dtype=np.float64)
    w = dfp["vz"].to_numpy(dtype=np.float64)
    o2 = dfp["O2"].to_numpy(dtype=np.float64)
    temp = dfp["temp"].to_numpy(dtype=np.float64)

    has_pres = "pres" in dfp.columns

    if has_pres:
        pres = dfp["pres"].to_numpy(dtype=np.float64)

    # ============================================================
    # 4. Reorganizar en ventanas
    # ============================================================

    hour_matrix = hour.reshape(n_blocks, n_samples)
    vx_matrix = vx.reshape(n_blocks, n_samples)
    vy_matrix = vy.reshape(n_blocks, n_samples)
    w_matrix = w.reshape(n_blocks, n_samples)
    o2_matrix = o2.reshape(n_blocks, n_samples)
    temp_matrix = temp.reshape(n_blocks, n_samples)

    if has_pres:
        pres_matrix = pres.reshape(n_blocks, n_samples)

    # ============================================================
    # 5. Definir rango de shifts
    # ============================================================

    if apply_shift:
        possible_shifts = range(
            shift_min_samples,
            shift_max_samples + 1
        )
    else:
        possible_shifts = [0]

    # Arrays donde guardaremos el resultado de cada ventana.
    #
    # fluxes_mmol_m2_s:
    #     flujo en mmol m⁻² s⁻¹
    #
    # optimal_shifts_samples:
    #     shift óptimo en número de muestras
    #
    # valid_flux_window:
    #     True si la ventana se ha podido procesar correctamente
    #
    # flux_status:
    #     texto que indica el estado de la ventana

    fluxes_mmol_m2_s = np.full(n_blocks, np.nan)
    optimal_shifts_samples = np.full(n_blocks, np.nan)
    valid_flux_window = np.full(n_blocks, False)
    flux_status = np.full(n_blocks, "", dtype=object)

    # ============================================================
    # 6. Procesar cada ventana de forma independiente
    # ============================================================
    # Se procesa ventana por ventana:
    #
    #   1. Se comprueba que la señal no esté plana.
    #   2. Se aplica detrending.
    #   3. Se busca el shift óptimo.
    #   4. Se calcula el flujo.

    for i in range(n_blocks):

        current_w = w_matrix[i, :]
        current_o2 = o2_matrix[i, :]

        # --------------------------------------------------------
        # 6.1. Señales planas
        # --------------------------------------------------------
        # Si w o O2 son prácticamente constantes en la ventana,
        # la covarianza no es informativa.
        #
        # Esto puede indicar sensor congelado, fallo de medida o una
        # ventana no útil para calcular flujo.

        if np.std(current_w) < 1e-12:
            flux_status[i] = "vz_plana"
            continue

        if np.std(current_o2) < 1e-12:
            flux_status[i] = "O2_plana"
            continue

        # --------------------------------------------------------
        # 6.2. Detrending lineal por ventana
        # --------------------------------------------------------
        # Eliminamos la tendencia lineal dentro de la ventana.
        #
        # w'  = fluctuación de la velocidad vertical
        # O2' = fluctuación de la concentración de oxígeno
        #
        # El flujo se calcula como la covarianza entre ambas señales.

        try:
            current_w_prime = signal.detrend(
                current_w,
                type="linear"
            )

            current_o2_prime = signal.detrend(
                current_o2,
                type="linear"
            )

        except Exception:
            flux_status[i] = "error_detrend"
            continue

        # --------------------------------------------------------
        # 6.3. Buscar el shift óptimo en la ventana
        # --------------------------------------------------------

        best_flux = np.nan
        best_score = -np.inf
        best_shift = np.nan

        for s in possible_shifts:

            # Flujo candidato para este shift.
            #
            # Unidades:
            #
            #     w'  -> m/s
            #     O2' -> mmol/m³
            #
            # Resultado:
            #
            #     mmol m⁻² s⁻¹

            flux_candidate = covarianza_con_shift(
                current_w_prime,
                current_o2_prime,
                s
            )

            if not np.isfinite(flux_candidate):
                continue

            # Usamos el valor absoluto para elegir el shift que maximiza
            # la magnitud de la covarianza.

            score = abs(flux_candidate)

            if score > best_score:
                best_score = score
                best_flux = flux_candidate
                best_shift = s

        # --------------------------------------------------------
        # 6.4. Guardar resultado de la ventana
        # --------------------------------------------------------

        if np.isfinite(best_flux):
            fluxes_mmol_m2_s[i] = best_flux
            optimal_shifts_samples[i] = best_shift
            valid_flux_window[i] = True
            flux_status[i] = "ok"
        else:
            flux_status[i] = "sin_flux_valido"

    # ============================================================
    # 7. Convertir flujo de segundos a días
    # ============================================================
    #
    # El flujo calculado sale en:
    #
    #     mmol m⁻² s⁻¹
    #
    # Para pasarlo a días:
    #
    #     flux_day = flux_second * 86400

    fluxes_mmol_m2_day = fluxes_mmol_m2_s * SECONDS_PER_DAY

    # Se calcula el flujo acumulado para la ventana en mmol m⁻²
    # Representa el total de O2 transferido por metro cuadrado en lo que dura la ventana.
    window_duration_seconds = window_minutes * 60
    accumulated_flux_mmol_m2 = fluxes_mmol_m2_s * window_duration_seconds

    # ============================================================
    # 8. Tiempo de cada ventana
    # ============================================================

    block_times_hour = np.mean(hour_matrix, axis=1)
    block_times_day = block_times_hour / HOURS_PER_DAY

    # ============================================================
    # 9. Convertir shift óptimo a segundos y días
    # ============================================================

    optimal_shifts_sec = optimal_shifts_samples / fs
    optimal_shifts_day = optimal_shifts_sec / SECONDS_PER_DAY

    # ============================================================
    # 10. Medias por ventana
    mean_vx = np.mean(vx_matrix, axis=1)
    mean_vy = np.mean(vy_matrix, axis=1)
    mean_vz = np.mean(w_matrix, axis=1)
    # Módulo medio de la velocidad por ventana: mean(sqrt(vx² + vy² + vz²))
    mean_speed = np.mean(np.sqrt(vx_matrix**2 + vy_matrix**2 + w_matrix**2), axis=1)
    mean_O2 = np.mean(o2_matrix, axis=1)
    mean_temp = np.mean(temp_matrix, axis=1)
    
    if has_pres:
        mean_pres = np.mean(pres_matrix, axis=1)
        std_pres = np.std(pres_matrix, axis=1)
    
    # ============================================================
    # 11. Crear DataFrame final
    # ============================================================

    data = {
        "window_id": np.arange(n_blocks),

        "hour": block_times_hour,
        "day": block_times_day,

        "flux_O2": fluxes_mmol_m2_day,
        "accumulated_flux_O2": accumulated_flux_mmol_m2,

        "optimal_shift_samples": optimal_shifts_samples,
        "optimal_shift_sec": optimal_shifts_sec,
        "optimal_shift_day": optimal_shifts_day,

        "mean_vx": mean_vx,
        "mean_vy": mean_vy,
        "mean_vz": mean_vz,
        "mean_speed": mean_speed,
        "mean_O2": mean_O2,
        "mean_temp": mean_temp,

        "valid_flux_window": valid_flux_window,
        "flux_status": flux_status
    }

    if has_pres:
        data["mean_pres"] = mean_pres
        data["std_pres"] = std_pres

    df_final = pd.DataFrame(data)

    # ============================================================
    # 12. Suavizado del flujo diario
    # ============================================================
    # El suavizado se aplica sobre flux_O2.
    #
    # Suavizado por media móvil.

    df_final["flux_smooth"] = (
        df_final["flux_O2"]
        .rolling(
            window=smooth_window,
            center=True,
            min_periods=1
        )
        .mean()
    )

    return df_final

# PROCESAMIENTO Y CREACIÓN DE RAW DATA

In [13]:
df_final = procesar_dataset_completo(
    df=df_clean
)

df_final_master = procesar_dataset_completo(
    df=df
)

# display(df_final.head())
# display(df_final["flux_status"].value_counts(dropna=False))

In [14]:
# --- FILTRADO DE DATOS ---
cutoff_hour = 178 # Hora aproximada donde comienza la caída en picado

df_filtered = df_final[df_final['hour'] < cutoff_hour].copy()
df_filtered_master = df_final_master[df_final_master['hour'] < cutoff_hour].copy()

In [15]:
output_path = os.path.join(BASE_PATH, "DATA", "CLEAN", "df_flux_clean.csv")
df_filtered.to_csv(output_path, index=False)
print(f"Datos guardados exitosamente en: {output_path}")

Datos guardados exitosamente en: g:\Mi unidad\TFM_NoeliaGarciaGarcia\Pipeline\DATA\CLEAN\df_flux_clean.csv


In [16]:
output_path = os.path.join(BASE_PATH, "DATA", "CLEAN", "df_flux_master.csv")
df_filtered_master.to_csv(output_path, index=False)
print(f"Datos guardados exitosamente en: {output_path}")

Datos guardados exitosamente en: g:\Mi unidad\TFM_NoeliaGarciaGarcia\Pipeline\DATA\CLEAN\df_flux_master.csv
